# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "02_asset_characterization_Momentum"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-08"
UPDATED = "2026-03-16"
PROJECT = "macro-market-analysis"
STAGE = "Research"
VERSION = "0.2.0"
GIT_HASH = ""  # completar con commit hash si se usa git
EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Compute momentum features for each asset in the universe.

This notebook calculates time-series momentum metrics using
rolling return aggregation across multiple horizons.

The resulting features are stored **per asset** in:

data/features/asset/Momentum/{asset}.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
# Updated: now each asset has its own processed parquet
INPUT_DATASETS = "data/processed/{asset}.parquet"

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = "data/features/asset/Momentum/{asset}.parquet"

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Momentum"

# ------------------------------------------------------------
# FEATURE DESCRIPTION
# ------------------------------------------------------------
FEATURE_DESCRIPTION = """
Momentum features measure the cumulative return of an asset
over multiple historical horizons.

These features capture trend persistence and are commonly used
in time-series momentum and cross-sectional momentum strategies.
"""

# ============================================================
# Load ASSET_UNIVERSE from registry
# ============================================================

from quant_research.data.registry.universe_registry import ASSET_UNIVERSE

# Extraer solo los símbolos para iterar
ASSETS_UNIVERSE = [asset.symbol for asset in ASSET_UNIVERSE]

print("Assets in universe:", ASSETS_UNIVERSE)

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "plotly >= 5.0", "pathlib"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
Features are computed independently per asset.

Cross-sectional transformations such as:
    - rankings
    - z-scores
    - dispersion
are handled later in:

03_systemic_features.ipynb

Incremental feature saving per asset is planned to allow
recomputing only missing or updated rows.
"""

# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

#-----------------------------
# Data manipulation
# -----------------------------
import pandas as pd  # pandas >= 2.0
import numpy as np   # numpy >= 1.25

# Optional: display settings for research notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None  # avoid SettingWithCopyWarning

# ============================================================
# IMPORT PATHS
# ============================================================
from quant_research.config.paths import PROCESSED_DATA_PATH, MOMENTUM_FEATURE_PATH

# ============================================================
# Example: load processed parquet for one asset
# ============================================================
asset = "SPY"
processed_file = PROCESSED_DATA_PATH / f"{asset}.parquet"

df = pd.read_parquet(processed_file)
#print(df.head())

# ============================================================
# Example: output feature path for saving
# ============================================================
output_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
print(output_file)

# -----------------------------
# Parquet engine
# -----------------------------
import pyarrow  # pyarrow >= 12.0

# -----------------------------
# Visualization
# -----------------------------
import plotly.express as px  # plotly >= 5.0
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -----------------------------
# Optional future imports from src/ once implemented
# -----------------------------
# from src.utils import panel_utils, plotting_utils

# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# -----------------------------
# Imports
# -----------------------------
from quant_research.config.paths import PROCESSED_DATA_PATH, MOMENTUM_FEATURE_PATH
from quant_research.data.registry.universe_registry import ASSET_UNIVERSE

# -----------------------------
# FEATURE FAMILY
# -----------------------------
FEATURE_FAMILY = "Momentum"

# -----------------------------
# FEATURE PARAMETERS
# -----------------------------
LOOKBACK_WINDOWS = [5,21, 63, 126, 252]

# Smoothing windows for velocity/acceleration derivatives (VEL_S, ACC_S)
SMOOTH_WINDOWS = {
    5: None,   # no smoothing para ultra-corto
    21: 3,
    63: 3,
    126: 5,
    252: 5
}

PERCENTILE_WINDOW = 252

NORMALIZATION_WINDOW = 252

MOM_ALIGN_THRESHOLD = 0.5

# -----------------------------
# UNIVERSE CONFIGURATION
# -----------------------------
# Use ASSET_UNIVERSE from registry
ASSETS = [asset.symbol for asset in ASSET_UNIVERSE]

# -----------------------------
# DATA PATHS
# -----------------------------
# Per-asset processed parquet
PROCESSED_DATA_DIR = PROCESSED_DATA_PATH

# Where to store features
FEATURE_OUTPUT_DIR = MOMENTUM_FEATURE_PATH

# -----------------------------
# EXPORT CONFIGURATION
# -----------------------------
EXPORT_FORMAT = "parquet"
OVERWRITE_EXISTING = True

# -----------------------------
# RESEARCH OPTIONS
# -----------------------------
# Sample assets for visualizations and sanity checks
SAMPLE_ASSETS_FOR_PLOTS = [
    "BTC",
    "SPY"
]

# -----------------------------
# RANDOM SEED
# -----------------------------
RANDOM_SEED = 42

# 4. Data Loading

In [ ]:
# ============================================================
# 4. DATA LOADING
# ============================================================

import pandas as pd
from pathlib import Path

# Diccionario para guardar los DataFrames por asset
asset_dfs = {}

print("Loading processed data for each asset...\n")

for asset in ASSETS:
    processed_file = PROCESSED_DATA_DIR / f"{asset}.parquet"
    
    if processed_file.exists():
        df = pd.read_parquet(processed_file)
        
        # -----------------------------
        # Validations
        # -----------------------------
        # Ensure datetime index
        if not pd.api.types.is_datetime64_any_dtype(df.index):
            try:
                df.index = pd.to_datetime(df.index)
            except Exception as e:
                raise ValueError(f"Asset {asset}: Cannot convert index to datetime") from e
        
        # Sort index ascending
        df = df.sort_index()
        
        # Drop duplicated indices
        df = df[~df.index.duplicated(keep="first")]
        
        # Save cleaned DataFrame
        asset_dfs[asset] = df
        
        print(f"Loaded {asset} → {len(df)} rows, {df.shape[1]} columns")
    
    else:
        print(f"WARNING: Processed parquet not found for {asset}: {processed_file}")
        
print("\nAll assets loaded. Ready for feature computation.")

# 5. Core Computation
MOMENTUM

Momentum represents the persistence of price movements over time.
Assets exhibiting positive momentum tend to continue outperforming,
while negative momentum assets tend to underperform.

Trend analysis allows us to quantify:

- Direction of movement
- Strength of persistence
- Stability of trends
- Speed of price evolution

Momentum features are fundamental inputs for both
asset allocation and alpha generation models.

In this section we construct statistical measures that
capture trend behavior across multiple horizons.

### 5.1 Multi-Horizon Momentum

Momentum measures the persistence of price trends over time.

Instead of relying on a single lookback window, we compute momentum
across multiple horizons to capture ultra-short-, short-, medium-, and long-term trends.

We compute momentum for each asset independently using **log returns**:

$$
Momentum(t, n) = \log \frac{P_t}{P_{t-n}}
$$

Where:

- $(P_t)$ = current adjusted price  
- $(P_{t-n})$ = price n periods ago  
- $(n)$ = lookback window  

Lookback horizons included:

- 5 days   → Ultra-short-term (~1 week)  
- 21 days  → Short-term (~1 month)  
- 63 days  → Medium-term (~3 months)  
- 126 days → Intermediate (~6 months)  
- 252 days → Long-term (~1 year)  

Optional smoothing can be applied using a rolling mean defined in `SMOOTH_WINDOWS`.  
Structural NaNs naturally appear during the warmup period and are preserved.

In [ ]:
# ============================================================
# 5.1 Multi-Horizon Momentum (asset-by-asset, incremental saving)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Iterate over each asset in the universe
for asset, df in asset_dfs.items():
    # File path for asset momentum features
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
    
    # Load existing momentum features if the parquet exists
    if asset_file.exists() and not OVERWRITE_EXISTING:
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}: {list(df_features.columns)}")
    else:
        # Start fresh with date index from processed prices
        df_features = pd.DataFrame(index=df.index)
        if asset_file.exists():
            print(f"Overwriting existing features for {asset}")
        else:
            print(f"Creating new feature file for {asset}")
    
    # Compute log momentum for each lookback window if not already in df_features
    for h in LOOKBACK_WINDOWS:
        col_name = f"MOM_{h}"
        if col_name not in df_features.columns:
            # Compute log momentum
            df_features[col_name] = np.log(df["adj_close"] / df["adj_close"].shift(h))
            print(f"Computed {col_name} for {asset}")
        else:
            print(f"{col_name} already exists for {asset}, skipping computation")
    
    # Save back to parquet (overwrite)
    df_features.to_parquet(asset_file)
    print(f"Saved features for {asset} → {asset_file}")

### 5.2 Momentum Stability

Momentum alone does not indicate whether a trend is reliable.

Momentum Stability measures how consistent momentum remains
through time. Stable momentum suggests persistent trends,
while unstable momentum indicates noisy or fragile price movements.

We estimate momentum stability as the rolling standard deviation
of momentum values.

Lower dispersion implies stronger trend persistence.

Mathematically:

$$ Momentum Stability = std(Momentum_t) over rolling window $$

Where:

- Low values → stable trend
- High values → unstable or reversing trend

This metric helps distinguish between sustainable trends
and short-lived market moves.

In [ ]:
# ============================================================
# 5.2 Momentum Stability (MOM_STAB_h)
# ============================================================

# Use the same rolling window as NORMALIZATION_WINDOW
STABILITY_WINDOW = NORMALIZATION_WINDOW

for asset, df in asset_dfs.items():
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"

    # Load or create features DataFrame
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")

    for h in LOOKBACK_WINDOWS:
        mom_col = f"MOM_{h}"
        stab_col = f"MOM_{h}_STAB"

        # Only compute if missing
        if stab_col not in df_features.columns and mom_col in df_features.columns:
            # Rolling standard deviation → lower values = more stable trend
            df_features[stab_col] = df_features[mom_col].rolling(
                STABILITY_WINDOW, min_periods=None
            ).std()

            print(f"Computed {stab_col} for {asset}")

    # Save back to parquet
    df_features.to_parquet(asset_file)
    print(f"Saved momentum stability features for {asset} → {asset_file}")

### 5.3 Momentum Z-Score (Normalized Trend Strength)

Raw momentum values are not directly comparable across assets,
as each asset exhibits different volatility and trend magnitude.

To normalize trend strength, momentum is standardized using
a rolling Z-score.

Z-score measures how extreme the current momentum is relative
to its historical distribution.

Mathematically:

$$ Z_t = (MOM_t - μ_t) / σ_t "" $$

Where:

$ - MOM_t $ : momentum at time t 
$ - μ_t $  : rolling mean of momentum 
$ - σ_t $ : rolling standard deviation 

Interpretation:

- Z > 2  → unusually strong trend
- Z ≈ 0  → normal trend conditions
- Z < -2 → unusually weak or bearish trend

This transformation allows comparison of trend strength
across assets and time.

In [ ]:
# ============================================================
# 5.2 Momentum Z-scores (asset-by-asset, incremental saving)
# ============================================================

# Iterate over each asset in the universe
for asset, df in asset_dfs.items():
    # File path for asset momentum features
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
    
    # Load existing features
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")
    
    # Compute Z-score for each momentum feature
    for h in LOOKBACK_WINDOWS:
        mom_col = f"MOM_{h}"
        z_col = f"MOM_{h}_Z"
        
        # Only compute if the Z-score column doesn't exist
        if z_col not in df_features.columns:
            if mom_col not in df_features.columns:
                # Compute momentum if missing
                df_features[mom_col] = np.log(df["adj_close"] / df["adj_close"].shift(h))
            
            # Rolling mean and std for Z-score
            rolling_mean = df_features[mom_col].rolling(NORMALIZATION_WINDOW).mean()
            rolling_std  = df_features[mom_col].rolling(NORMALIZATION_WINDOW).std()
            
            # Compute Z-score
            df_features[z_col] = (df_features[mom_col] - rolling_mean) / rolling_std
            print(f"Computed {z_col} for {asset}")
        else:
            print(f"{z_col} already exists for {asset}, skipping")
    
    # Save back to parquet (overwrite)
    df_features.to_parquet(asset_file)
    print(f"Saved features for {asset} → {asset_file}")

#### 5.3.1 Momentum Percentile

The **Momentum Percentile** measures the relative position of the current momentum value within its own historical distribution.

For each asset and lookback window, we compute the rolling percentile rank of the momentum level over a specified historical window.

This transformation provides a **distribution-free normalization** of momentum, allowing us to identify whether the current trend is weak, neutral, or extreme relative to the asset’s own history.

Mathematically: $$ MOM_{w,PCTL,t}=\frac{rank\left(MOM_{w,t} \;|\; MOM_{w,t-L+1}, ..., MOM_{w,t}\right)}{L} $$

Where:

- $w$ = momentum lookback window (e.g. 21, 63, 126, 252)
- $L$ = percentile window length
- $rank(\cdot)$ = rank of the current value within the rolling window

Interpretation:

| Percentile | Interpretation |
|---|---|
| ~0.0 – 0.2 | Very weak momentum |
| ~0.4 – 0.6 | Neutral momentum |
| ~0.8 – 1.0 | Strong / extreme momentum |

Unlike Z-scores, percentile normalization does **not assume a specific distribution**, making it more robust to outliers and heavy-tailed return behavior.

These features are labeled:

```
MOM_21_PCTL
MOM_63_PCTL
MOM_126_PCTL
MOM_252_PCTL
```

and represent **momentum regimes relative to each asset's own historical behavior**.

In [ ]:
# ============================================================
# 5.3 Momentum Percentiles (asset-by-asset, incremental saving)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Iterate over each asset in the universe
for asset, df in asset_dfs.items():
    # File path for asset momentum features
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
    
    # Load existing features if they exist
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")
    
    # Compute rolling percentiles for each momentum feature
    for h in LOOKBACK_WINDOWS:
        mom_col = f"MOM_{h}"
        pct_col = f"MOM_{h}_PCTL"
        
        # Only compute if the percentile column doesn't exist
        if pct_col not in df_features.columns:
            if mom_col not in df_features.columns:
                # Compute momentum if missing
                df_features[mom_col] = np.log(df["adj_close"] / df["adj_close"].shift(h))
            
            # Rolling percentile (0-1) over PERCENTILE_WINDOW, preserve NaNs for warmup
            df_features[pct_col] = df_features[mom_col].rolling(
                PERCENTILE_WINDOW, min_periods=None
            ).rank(pct=True)
            
            print(f"Computed {pct_col} for {asset}")
        else:
            print(f"{pct_col} already exists for {asset}, skipping")
    
    # Save back to parquet (overwrite)
    df_features.to_parquet(asset_file)
    print(f"Saved features for {asset} → {asset_file}")

### 5.4 Momentum Agreement Index

Momentum signals across different horizons may diverge.

The Momentum Agreement Index measures whether short-, medium-,
and long-term momentum signals point in the same direction.

Trend conviction increases when multiple horizons agree.

We define agreement as the average sign of momentum signals:

$$ Agreement = mean(sign(Momentum_i)) $$

Where Momentum_i represents momentum computed
across multiple lookback windows.

Interpretation:

- +1 → Strong bullish agreement
- 0  → Mixed or transition regime
- -1 → Strong bearish agreement

This metric helps detect high-conviction trends
and potential regime shifts.

In [ ]:
# ============================================================
# 5.5 Momentum Alignment (MOM_ALIGN & MOM_ALIGN_Z)
# ============================================================

for asset, df in asset_dfs.items():
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
    
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")
    
    # Identify columns
    mom_cols = [f"MOM_{h}" for h in LOOKBACK_WINDOWS if f"MOM_{h}" in df_features.columns]
    mom_Z_cols = [f"MOM_{h}_Z" for h in LOOKBACK_WINDOWS if f"MOM_{h}_Z" in df_features.columns]
    
    # --------------------------------------------------------
    # MOM_ALIGN (sin threshold)
    # --------------------------------------------------------
    if "MOM_ALIGN" not in df_features.columns and mom_cols:
        df_features["MOM_ALIGN"] = np.sign(df_features[mom_cols]).mean(axis=1)
        print(f"Computed MOM_ALIGN for {asset}")
    
    # --------------------------------------------------------
    # MOM_ALIGN_Z (threshold sobre MOM_h_Z)
    # --------------------------------------------------------
    if "MOM_ALIGN_Z" not in df_features.columns and mom_Z_cols:
        
        df_z = df_features[mom_Z_cols].copy()
        
        # 🔑 CLAVE: excluir ruido → NaN (NO 0)
        df_z[np.abs(df_z) < MOM_ALIGN_THRESHOLD] = np.nan
        
        # sign
        signs = np.sign(df_z)
        
        # 🔑 promedio solo sobre válidos
        mom_align_thresh = signs.sum(axis=1) / signs.notna().sum(axis=1)
        
        df_features["MOM_ALIGN_Z"] = mom_align_thresh
        
        print(f"Computed MOM_ALIGN_Z with threshold {MOM_ALIGN_THRESHOLD} for {asset}")
    
    df_features.to_parquet(asset_file)
    print(f"Saved momentum alignment features for {asset} → {asset_file}")

## 5.4.1 Momentum Agreement Index (Z-Score Based)

Objective

Measure multi-horizon trend alignment using standardized momentum (Z-scores) 
instead of raw momentum values.

While raw momentum agreement captures directional alignment, it treats weak 
and strong signals equally. Using Z-scores allows us to account for statistical 
significance relative to historical behavior.

---

Economic Intuition

Markets exhibit stronger regime behavior when multiple time horizons 
show statistically significant trend alignment.

If:

- Short-term momentum is positive,
- Medium-term momentum is positive,
- Long-term momentum is positive,

and all are meaningfully different from zero (high |Z|),

then the probability of a persistent bullish regime increases.

The same logic applies for bearish alignment.

---

Construction

For each asset and date:

1. Compute the sign of standardized momentum (Z-score).
2. Average the signs across all horizons.
3. Optionally filter weak signals (|Z| below threshold).

The resulting index lies in:

- +1  → all horizons significantly bullish
-  0  → mixed or weak signals
- -1  → all horizons significantly bearish

---

Interpretation

This feature represents a:

Multi-Scale Significant Trend Alignment Score

It can later be used for:
- Regime detection
- Signal filtering
- Conditional allocation rules

## 5.5 Momentum Dynamics


In [ ]:
# ============================================================
# 5.4 Momentum Dynamics (Velocity & Acceleration)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

for asset, df in asset_dfs.items():
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"
    
    # Load or create features DataFrame
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")
    
    for h in LOOKBACK_WINDOWS:
        mom_col = f"MOM_{h}"
        vel_col = f"MOM_{h}_VEL"
        acc_col = f"MOM_{h}_ACC"
        
        # Compute momentum if missing
        if mom_col not in df_features.columns:
            df_features[mom_col] = np.log(df["adj_close"] / df["adj_close"].shift(h))
            print(f"Computed {mom_col} for {asset}")
        
        # Compute velocity
        if vel_col not in df_features.columns:
            df_features[vel_col] = df_features[mom_col].diff()
            print(f"Computed {vel_col} for {asset}")
        
        # Compute acceleration
        if acc_col not in df_features.columns:
            df_features[acc_col] = df_features[vel_col].diff()
            print(f"Computed {acc_col} for {asset}")
        
        # Determine smoothing window
        smooth_win = SMOOTH_WINDOWS.get(h, 3)
        
        # Only compute smoothed features if smooth_win is defined
        if smooth_win is not None:
            vel_s_col = f"MOM_{h}_VEL_S"
            acc_s_col = f"MOM_{h}_ACC_S"
            
            if vel_s_col not in df_features.columns:
                df_features[vel_s_col] = df_features[vel_col].rolling(
                    smooth_win, min_periods=None
                ).mean()
                print(f"Computed {vel_s_col} for {asset}")
            
            if acc_s_col not in df_features.columns:
                df_features[acc_s_col] = df_features[acc_col].rolling(
                    smooth_win, min_periods=None
                ).mean()
                print(f"Computed {acc_s_col} for {asset}")
        else:
            # No smoothing → skip VEL_S / ACC_S for this horizon
            print(f"No smoothing applied for horizon {h} ({asset})")
    
    # Save back to parquet
    df_features.to_parquet(asset_file)
    print(f"Saved momentum dynamics features for {asset} → {asset_file}")

## 5.6 Momentum Strength Index (Weighted & Smoothed)

Individual momentum horizons capture trend dynamics at
different temporal scales. However, analyzing them
separately does not provide a unified structural view
of market strength.

To aggregate cross-horizon information, we define the
Momentum Strength Index (MSI) as a weighted combination
of normalized momentum signals.

Let $ Z_t(h) $ denote the Z-scored momentum for horizon h.

The weighted MSI is defined as:

$$ MSI_t = Σ_h ( w_h * Z_t(h) ) $$

where weights satisfy:

$$ Σ_h w_h = 1 $$

The weighting scheme reflects structural importance
across horizons.

In our baseline specification:

w_21  = 0.1  
w_63  = 0.2  
w_126 = 0.3  
w_252 = 0.4  

Longer horizons receive higher weights to emphasize
structural regime components over tactical noise.

Since MSI aggregates multiple horizons, it is further
smoothed to enhance regime stability:

$$ M̃SI_t = (1 / k) * Σ_{i=0}^{k-1} MSI_{t-i} $$

with k = 5.

This produces a structurally stable, cross-horizon
trend-strength indicator suitable for regime detection
and allocation modeling.

In [ ]:
# ============================================================
# 5.6 Momentum Strength Index (MSI)
# ============================================================

# Define weights for each horizon
MSI_WEIGHTS = {
    5: 0.0,
    21: 0.1,
    63: 0.2,
    126: 0.3,
    252: 0.4
}

# Smoothing window for MSI
MSI_SMOOTH_WINDOW = 5

for asset, df in asset_dfs.items():
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"

    # Load or create features DataFrame
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")

    # Only compute MSI if missing
    if "MSI" not in df_features.columns:
        # Identify available MOM_h_Z columns
        z_cols = [f"MOM_{h}_Z" for h in LOOKBACK_WINDOWS if f"MOM_{h}_Z" in df_features.columns]
        if not z_cols:
            print(f"No Z-scored momentum columns found for {asset}, skipping MSI")
            continue

        # Apply weights only to available horizons
        weights = np.array([MSI_WEIGHTS[int(col.split("_")[1])] for col in z_cols])
        weights = weights / weights.sum()  # normalize to sum=1

        # Compute weighted sum across horizons → MSI
        df_features["MSI"] = df_features[z_cols].dot(weights)

        # Smooth MSI with simple rolling mean
        df_features["MSI_S"] = df_features["MSI"].rolling(MSI_SMOOTH_WINDOW, min_periods=None).mean()

        print(f"Computed MSI and MSI_S for {asset}")

    # Save back to parquet
    df_features.to_parquet(asset_file)
    print(f"Saved MSI features for {asset} → {asset_file}")

## 5.6.1 MSI Velocity (Smoothed)

To capture structural changes in aggregated trend strength,
we compute the first derivative of the smoothed MSI.

MSI velocity is defined as:

$$ MSI_{Vel_t} = M̃SI_t - M̃SI_{t-1} $$

This measures the rate of change in structural momentum
across horizons.

Because MSI is already smoothed, velocity reflects
structural acceleration rather than tactical noise.

To ensure stability, MSI velocity is additionally
smoothed using a 5-day rolling mean:

$$ M̃SI_{Vel_t} = (1 / 5) * Σ_{i=0}^{4} MSI_{Vel_{t-i}} $$

This produces a stable structural force indicator,
useful for identifying regime transitions.

In [ ]:
# ============================================================
# 5.6.1 MSI Velocity (MSI_VEL and MSI_VEL_S)
# ============================================================

for asset, df in asset_dfs.items():
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"

    # Load or create features DataFrame
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")

    # Ensure MSI_S exists
    if "MSI_S" not in df_features.columns:
        print(f"MSI_S not found for {asset}, skipping MSI_VEL")
        continue

    # ------------------------------------------------------------
    # Compute MSI velocity
    # ------------------------------------------------------------
    if "MSI_VEL" not in df_features.columns:
        df_features["MSI_VEL"] = df_features["MSI_S"].diff()
        print(f"Computed MSI_VEL for {asset}")

    # ------------------------------------------------------------
    # Smoothed MSI velocity
    # ------------------------------------------------------------
    if "MSI_VEL_S" not in df_features.columns:
        df_features["MSI_VEL_S"] = df_features["MSI_VEL"].rolling(
            MSI_SMOOTH_WINDOW, min_periods=None
        ).mean()
        print(f"Computed MSI_VEL_S for {asset}")

    # Save features
    df_features.to_parquet(asset_file)
    print(f"Saved MSI velocity features for {asset} → {asset_file}")

## 5.6.2 MSI Acceleration (Smoothed)

To detect structural inflection points, we compute
the second derivative of the smoothed MSI.

MSI acceleration is defined as:

$$ MSI_{Acc_t} = MSI_{Vel_t} - MSI_{Vel_{t-1}} $$

Equivalently:

$$ MSI_{Acc_t} = Δ² M̃SI_t $$

Acceleration captures changes in structural momentum force,
identifying reinforcement or exhaustion of dominant regimes.

Given the noise amplification inherent in second derivatives,
MSI acceleration is smoothed using a 5-day rolling mean:

$$ M̃SI_{Acc_t} = (1 / 5) * Σ_{i=0}^{4} MSI_{Acc_{t-i}} $$

The resulting signal provides a stable structural
inflection indicator suitable for regime-state modeling.

In [ ]:
# ============================================================
# 5.6.2 MSI Acceleration (MSI_ACC and MSI_ACC_S)
# ============================================================

for asset, df in asset_dfs.items():
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"

    # Load or create features DataFrame
    if asset_file.exists():
        df_features = pd.read_parquet(asset_file)
        print(f"Loaded existing features for {asset}")
    else:
        df_features = pd.DataFrame(index=df.index)
        print(f"Creating new feature file for {asset}")

    # Ensure MSI_VEL exists
    if "MSI_VEL" not in df_features.columns:
        print(f"MSI_VEL not found for {asset}, skipping MSI_ACC")
        continue

    # ------------------------------------------------------------
    # Compute MSI acceleration
    # ------------------------------------------------------------
    if "MSI_ACC" not in df_features.columns:
        df_features["MSI_ACC"] = df_features["MSI_VEL"].diff()
        print(f"Computed MSI_ACC for {asset}")

    # ------------------------------------------------------------
    # Smoothed MSI acceleration
    # ------------------------------------------------------------
    if "MSI_ACC_S" not in df_features.columns:
        df_features["MSI_ACC_S"] = df_features["MSI_ACC"].rolling(
            MSI_SMOOTH_WINDOW, min_periods=None
        ).mean()
        print(f"Computed MSI_ACC_S for {asset}")

    # Save back to parquet
    df_features.to_parquet(asset_file)
    print(f"Saved MSI acceleration features for {asset} → {asset_file}")

In [ ]:
df_features.columns

# 6. Diagnosis/Validation

## 6.1 Dataset integrity checks

In [ ]:
# ============================================================
# 6.1 Data Integrity Checks
# ============================================================

for asset in ASSETS:
    df_stored = pd.read_parquet(MOMENTUM_FEATURE_PATH / f"{asset}.parquet")
    
    print(f"\nAsset: {asset}")
    print("Shape:", df_features.shape)
    print("Date range:", df.index.min(), "→", df.index.max())
    print("NaNs per column:")
    print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
df_stored.columns

## 6.2 Feature Correctenes Checks



In [ ]:
df.columns

In [ ]:
from quant_research.features.asset.momentum.feature_validator import validate_universe_features

summary = validate_universe_features(
    asset_dfs=asset_dfs,
    feature_path=MOMENTUM_FEATURE_PATH,
    verbose=False,
)

display(summary)

# 7. Visualization

In [ ]:


asset = "SPY"

df_stored = pd.read_parquet(MOMENTUM_FEATURE_PATH / f"{asset}.parquet")
df = pd.read_parquet(PROCESSED_DATA_PATH / f"{asset}.parquet")



In [ ]:
cols = ["MOM_21", "MOM_21_Z", "MOM_21_PCTL"]

fig = go.Figure()

for col in cols:
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df_stored[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset} - Momentum Normalizations",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df_stored["MSI"],
    mode="lines",
    name="MSI",
    line=dict(width=1)
))

fig.add_trace(go.Scatter(
    x=df.index,
    y=df_stored["MSI_S"],
    mode="lines",
    name="MSI_S",
    line=dict(width=2)
))

fig.update_layout(
    title=f"{asset} - MSI",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_stored.index,
    y=df_stored["MOM_ALIGN"],
    mode="lines",
    name="MOM_ALIGN"
))

fig.add_trace(go.Scatter(
    x=df_stored.index,
    y=df_stored["MOM_ALIGN_Z"],
    mode="lines",
    name="MOM_ALIGN_Z"
))

fig.update_layout(
    title=f"{asset} - Momentum Alignment",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
cols = ["MOM_21_VEL", "MOM_21_VEL_S", "MOM_21_ACC", "MOM_21_ACC_S"]

fig = go.Figure()

for col in cols:
    fig.add_trace(
        go.Scatter(
            x=df_stored.index,
            y=df_stored[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title=f"{asset} - Derivatives",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
feature = "MSI_S"  # podés cambiar a MOM_21_PCTL, MOM_ALIGN_Z, etc.

fig = make_subplots(specs=[[{"secondary_y": True}]])

# -------------------------
# PRICE (izquierda)
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["adj_close"],
        name="PRICE",
        line=dict(color="white", width=2)
    ),
    secondary_y=False
)

# -------------------------
# FEATURE (derecha)
# -------------------------
fig.add_trace(
    go.Scatter(
        x=df_stored.index,
        y=df_stored[feature],
        name=feature,
        line=dict(color="cyan", width=2)
    ),
    secondary_y=True
)

fig.update_layout(
    title=f"{asset} - Price vs {feature}",
    template="plotly_dark",
    height=600
)

fig.update_yaxes(title_text="Price", secondary_y=False)
fig.update_yaxes(title_text=feature, secondary_y=True)

fig.show()

In [ ]:
features = ["MSI", "MOM_21_PCTL", "MOM_ALIGN_Z"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Price
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["adj_close"],
        name="PRICE",
        line=dict(color="white", width=2)
    ),
    secondary_y=False
)

colors = ["cyan", "orange", "green"]

for f, c in zip(features, colors):
    fig.add_trace(
        go.Scatter(
            x=df_stored.index,
            y=df_stored[f],
            name=f,
            line=dict(color=c, width=1.5)
        ),
        secondary_y=True
    )

fig.update_layout(
    title=f"{asset} - Price vs Multiple Features",
    template="plotly_dark",
    height=600
)

fig.show()

# 8. Orquestation SRC (STAGE 2)

from quant_research.features.asset.momentum.features_engine import compute_momentum_features

from quant_research.features.asset.momentum.config import (
    LOOKBACK_WINDOWS,
    NORMALIZATION_WINDOW,
    SMOOTH_WINDOWS,
    MSI_WEIGHTS,
    MOM_ALIGN_THRESHOLD,
    MSI_SMOOTH_WINDOW,
)

for asset, df_raw in asset_dfs.items():
    
    print(f"\nProcessing {asset}...")

    # -------------------------
    # Compute features (ENGINE)
    # -------------------------
    df_features = compute_momentum_features(
        df_raw=df_raw,
        lookback_windows=LOOKBACK_WINDOWS,
        normalization_window=NORMALIZATION_WINDOW,
        smooth_windows=SMOOTH_WINDOWS,
        msi_weights=MSI_WEIGHTS,
        mom_align_threshold=MOM_ALIGN_THRESHOLD,
        msi_smooth_window=MSI_SMOOTH_WINDOW,
    )

    # -------------------------
    # Save
    # -------------------------
    asset_file = MOMENTUM_FEATURE_PATH / f"{asset}.parquet"

    df_features.to_parquet(asset_file)   #descomentar en refactor Stage 2 

    print(f"Saved → {asset_file}")

from quant_research.features.asset.momentum.feature_validator import validate_universe_features

summary = validate_universe_features(
    asset_dfs=asset_dfs,
    feature_path=MOMENTUM_FEATURE_PATH,
    verbose=False,
)

display(summary)

In [ ]:
def sign_changes(series: pd.Series) -> int:
    s = np.sign(series.dropna())
    return (s != s.shift(1)).sum()

In [ ]:
def analyze_smoothing(df, raw_col, smooth_col):
    
    raw = df[raw_col].dropna()
    smooth = df[smooth_col].dropna()

    # align index
    common_idx = raw.index.intersection(smooth.index)
    raw = raw.loc[common_idx]
    smooth = smooth.loc[common_idx]

    return {
        "feature": raw_col,
        "var_raw": raw.var(),
        "var_smooth": smooth.var(),
        "var_reduction_%": 100 * (1 - smooth.var() / raw.var()),
        "corr": raw.corr(smooth),
        "sign_changes_raw": sign_changes(raw),
        "sign_changes_smooth": sign_changes(smooth),
        "sign_reduction_%": 100 * (
            1 - sign_changes(smooth) / sign_changes(raw)
        )
    }

In [ ]:
results = []

pairs = [
    ("MSI", "MSI_S"),
    ("MSI_VEL", "MSI_VEL_S"),
    ("MSI_ACC", "MSI_ACC_S"),
    ("MOM_21_VEL", "MOM_21_VEL_S"),
    ("MOM_21_ACC", "MOM_21_ACC_S"),
]

for raw, smooth in pairs:
    results.append(analyze_smoothing(df_stored, raw, smooth))

df_diag = pd.DataFrame(results)
df_diag

In [ ]:
df_diag.sort_values("var_reduction_%", ascending=False)

In [ ]:
df_diag["signal_quality"] = (
    df_diag["corr"] * df_diag["var_reduction_%"]
)
print(df_diag["signal_quality"])

In [ ]:
EVAL_CONFIG = {
    "var_reduction_min": 0.2,
    "var_reduction_max": 0.9,
    "corr_min": 0.4,
    "sign_reduction_min": 0.2,
    "nan_ratio_max": 0.3,
}

In [ ]:
def evaluate_feature_pair(df, raw_col, smooth_col, config):

    raw = df[raw_col].dropna()
    smooth = df[smooth_col].dropna()

    idx = raw.index.intersection(smooth.index)
    raw = raw.loc[idx]
    smooth = smooth.loc[idx]

    # -------------------------
    # Metrics
    # -------------------------
    var_raw = raw.var()
    var_smooth = smooth.var()
    var_reduction = 1 - var_smooth / var_raw

    corr = raw.corr(smooth)

    def sign_changes(x):
        s = np.sign(x)
        return (s != s.shift(1)).sum()

    sc_raw = sign_changes(raw)
    sc_smooth = sign_changes(smooth)
    sign_reduction = 1 - sc_smooth / sc_raw

    nan_ratio = df[smooth_col].isna().mean()

    # -------------------------
    # Rules
    # -------------------------
    checks = {
        "var_reduction": config["var_reduction_min"] <= var_reduction <= config["var_reduction_max"],
        "correlation": corr >= config["corr_min"],
        "sign_reduction": sign_reduction >= config["sign_reduction_min"],
        "nan_ratio": nan_ratio <= config["nan_ratio_max"],
    }

    # -------------------------
    # Status
    # -------------------------
    if all(checks.values()):
        status = "PASS"
    elif checks["correlation"] and checks["var_reduction"]:
        status = "WARNING"
    else:
        status = "FAIL"

    return {
        "feature": raw_col,
        "var_reduction": var_reduction,
        "corr": corr,
        "sign_reduction": sign_reduction,
        "nan_ratio": nan_ratio,
        "status": status,
        "checks": checks,
    }

In [ ]:
def evaluate_feature_set(df, pairs, config):

    results = []

    for raw, smooth in pairs:
        res = evaluate_feature_pair(df, raw, smooth, config)
        results.append(res)

    return pd.DataFrame(results)

In [ ]:
pairs = [
    ("MSI", "MSI_S"),
    ("MSI_VEL", "MSI_VEL_S"),
    ("MSI_ACC", "MSI_ACC_S"),
    ("MOM_21_VEL", "MOM_21_VEL_S"),
    ("MOM_21_ACC", "MOM_21_ACC_S"),
]

df_eval = evaluate_feature_set(df_stored, pairs, EVAL_CONFIG)

df_eval[["feature", "status", "var_reduction", "corr", "sign_reduction"]]